In [1]:
import os

In [2]:
%pwd

'c:\\Users\\Hp\\Desktop\\cifar10-image-classificatio\\research'

In [3]:
os.chdir("../")

In [24]:
import numpy as np
from pathlib import Path

In [18]:
from dataclasses import dataclass
from pathlib import Path
from typing import Tuple


@dataclass
class TrainingConfig:
    root_dir: Path
    trained_model_path: Path
    updated_base_model_path: Path
    training_data: Path

    epochs: int
    batch_size: int
    is_augmentation: bool
    image_size: Tuple[int, int, int]

    learning_rate: float
    weight_decay: float

    rotation_range: int
    width_shift_range: float
    height_shift_range: float
    horizontal_flip: bool
    zoom_range: float
    brightness_range: tuple
    shear_range: float
    channel_shift_range: float

    reduce_lr_factor: float
    reduce_lr_patience: int
    min_learning_rate: float
    early_stopping_patience: int

In [ ]:
from src.cnnClassifier.constants import *
from src.cnnClassifier.utils.common import read_yaml, create_directories
import tensorflow as tf

In [ ]:
from pathlib import Path
from src.cnnClassifier.entity import TrainingConfig
from src.cnnClassifier.utils.common import create_directories, read_yaml


class ConfigurationManager:
    def __init__(self, config_filepath, params_filepath):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_training_config(self):

        training = self.config.training
        prepare_model = self.config.prepare_model
        params = self.params

        create_directories([Path(training.root_dir)])

        return TrainingConfig(

            root_dir=Path(training.root_dir),
            trained_model_path=Path(training.trained_model_path),
            updated_base_model_path=Path(prepare_model.model_path),

            # 🔥 IMPORTANT CHANGE (NPY DATA)
            training_data=Path("artifacts/data_ingestion"),

            epochs=int(params.EPOCHS),
            batch_size=int(params.BATCH_SIZE),
            image_size=tuple(params.IMAGE_SIZE),

            is_augmentation=str(params.AUGMENTATION).lower() == "true",

            learning_rate=float(params.LEARNING_RATE),
            weight_decay=float(params.WEIGHT_DECAY),

            rotation_range=int(params.ROTATION_RANGE),
            width_shift_range=float(params.WIDTH_SHIFT_RANGE),
            height_shift_range=float(params.HEIGHT_SHIFT_RANGE),
            horizontal_flip=str(params.HORIZONTAL_FLIP).lower() == "true",
            zoom_range=float(params.ZOOM_RANGE),
            brightness_range=tuple(params.BRIGHTNESS_RANGE),
            shear_range=float(params.SHEAR_RANGE),
            channel_shift_range=float(params.CHANNEL_SHIFT_RANGE),

            reduce_lr_factor=float(params.REDUCE_LR_FACTOR),
            reduce_lr_patience=int(params.REDUCE_LR_PATIENCE),
            min_learning_rate=float(params.MIN_LEARNING_RATE),
            early_stopping_patience=int(params.EARLY_STOPPING_PATIENCE)
        )

In [21]:
import tensorflow as tf
from pathlib import Path


class Training:
    def __init__(self, config: TrainingConfig):
        self.config = config

    def get_base_model(self):
        self.model = tf.keras.models.load_model(
            self.config.updated_base_model_path
        )

    def train_valid_generator(self):

        datagen_kwargs = dict(
            rescale=1./255,
            validation_split=0.20
        )

        flow_kwargs = dict(
            target_size=self.config.image_size[:-1],
            batch_size=self.config.batch_size,
            interpolation="bilinear"
        )

        valid_datagen = tf.keras.preprocessing.image.ImageDataGenerator(
            **datagen_kwargs
        )

        self.valid_generator = valid_datagen.flow_from_directory(
            directory=self.config.training_data,
            subset="validation",
            shuffle=False,
            **flow_kwargs
        )

        if self.config.is_augmentation:
            train_datagen = tf.keras.preprocessing.image.ImageDataGenerator(
                rotation_range=self.config.rotation_range,
                width_shift_range=self.config.width_shift_range,
                height_shift_range=self.config.height_shift_range,
                horizontal_flip=self.config.horizontal_flip,
                zoom_range=self.config.zoom_range,
                brightness_range=self.config.brightness_range,
                shear_range=self.config.shear_range,
                channel_shift_range=self.config.channel_shift_range,
                **datagen_kwargs
            )
        else:
            train_datagen = valid_datagen

        self.train_generator = train_datagen.flow_from_directory(
            directory=self.config.training_data,
            subset="training",
            shuffle=True,
            **flow_kwargs
        )

    def get_callbacks(self):

        return [
            tf.keras.callbacks.ReduceLROnPlateau(
                monitor="val_loss",
                factor=self.config.reduce_lr_factor,
                patience=self.config.reduce_lr_patience,
                min_lr=self.config.min_learning_rate,
                verbose=1
            ),

            tf.keras.callbacks.EarlyStopping(
                monitor="val_loss",
                patience=self.config.early_stopping_patience,
                restore_best_weights=True,
                verbose=1
            )
        ]

    def train(self):

        self.train_valid_generator()
        self.get_base_model()

        steps_per_epoch = self.train_generator.samples // self.train_generator.batch_size
        validation_steps = self.valid_generator.samples // self.valid_generator.batch_size

        optimizer = tf.keras.optimizers.Adam(
            learning_rate=self.config.learning_rate
        )

        self.model.compile(
            optimizer=optimizer,
            loss="categorical_crossentropy",
            metrics=["accuracy"]
        )

        self.model.fit(
            self.train_generator,
            validation_data=self.valid_generator,
            epochs=self.config.epochs,
            steps_per_epoch=steps_per_epoch,
            validation_steps=validation_steps,
            callbacks=self.get_callbacks(),
            verbose=2
        )

        self.model.save(self.config.trained_model_path)

In [23]:
config_manager = ConfigurationManager(
    config_filepath=CONFIG_FILE_PATH,
    params_filepath=PARAMS_FILE_PATH
)

training_config = config_manager.get_training_config()

training = Training(training_config)
training.train()

[2026-07-01 11:06:46,788: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-07-01 11:06:46,798: INFO: common: yaml file: params.yaml loaded successfully]
[2026-07-01 11:06:46,798: INFO: common: created directory at: artifacts]
[2026-07-01 11:06:46,804: INFO: common: created directory at: artifacts\training]
Found 0 images belonging to 0 classes.
Found 0 images belonging to 0 classes.
Unexpected exception formatting exception. Falling back to standard exception


Traceback (most recent call last):
  File "c:\Users\Hp\anaconda3\envs\cnncls\lib\site-packages\IPython\core\interactiveshell.py", line 3508, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "C:\Users\Hp\AppData\Local\Temp\ipykernel_3144\1196848210.py", line 9, in <module>
    training.train()
  File "C:\Users\Hp\AppData\Local\Temp\ipykernel_3144\2076794661.py", line 97, in train
    self.model.fit(
  File "c:\Users\Hp\anaconda3\envs\cnncls\lib\site-packages\keras\src\utils\traceback_utils.py", line 70, in error_handler
    raise e.with_traceback(filtered_tb) from None
  File "c:\Users\Hp\anaconda3\envs\cnncls\lib\site-packages\keras\src\preprocessing\image.py", line 103, in __getitem__
    raise ValueError(
ValueError: Asked to retrieve element 0, but the Sequence has length 0

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "c:\Users\Hp\anaconda3\envs\cnncls\lib\site-packages\IPython\core\interactiv